# Customer Segmentation Assignment
Here is my solution for the 5 tasks.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Task 1: Load data
df = pd.read_csv('q2_customers.csv')

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)

# just testing to make sure it scaled right

**Why do we scale?**
Scaling is essential because K-means relies on distances between points. If we don't scale, big numbers like `annual_spend` (in the tens of thousands) will completely overshadow small things like `visits_per_month`. 

In [ ]:
from sklearn.cluster import KMeans

# Task 2: Elbow method
wcss_list = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, random_state=10)
    km.fit(X_scaled)
    wcss_list.append(km.inertia_)

plt.plot(range(1, 11), wcss_list, marker='o')
plt.title('Elbow Graph to find K')
plt.xlabel('Number of clusters')
plt.ylabel('WCSS')
plt.show()

I am picking K=3. Looking at the elbow plot, 3 is where the curve starts flattening out and the big drops stop.

In [ ]:
# Task 3: K-Means
kmeans = KMeans(n_clusters=3, random_state=10)
df['cluster_label'] = kmeans.fit_predict(X_scaled)

# get original values back for centroids
cents = scaler.inverse_transform(kmeans.cluster_centers_)
cents_df = pd.DataFrame(cents, columns=['age', 'annual_spend', 'visits_per_month', 'basket_size', 'days_since_last_visit', 'num_categories_purchased'])
print("Centroids:")
print(cents_df)

**Business meaning of clusters:**
- Cluster 0: Older customers (~51 yrs) with very high annual spend, but they only visit like twice a month. Basically VIPs / Whales.
- Cluster 1: Young people (~25 yrs) who spend way less per year, but they visit all the time (14 times/month). Frequent budget shoppers.
- Cluster 2: Middle-aged (39 yrs), average spend, average visits. The regular mid-tier folks.

In [ ]:
from sklearn.decomposition import PCA

# Task 4: PCA
pca = PCA(n_components=2)
pca_data = pca.fit_transform(X_scaled)

print("Explained variance:", pca.explained_variance_ratio_)

# getting the loadings
loadings = pd.DataFrame(pca.components_, columns=['age', 'annual_spend', 'visits_per_month', 'basket_size', 'days_since_last_visit', 'num_categories_purchased'])
print("\nFeature Loadings:")
print(loadings)

PC1 seems to separate people based on spending vs frequency (it has big positive numbers for spend/basket size and big negative numbers for visits).
PC2 is mostly just capturing age and a little bit of the days since last visit.

In [ ]:
# Task 5: Viz
# saving pc1 and pc2 into the main df to make plotting easier
df['pc1'] = pca_data[:, 0]
df['pc2'] = pca_data[:, 1]

plt.figure(figsize=(8,5))

# loop through to get the legend working
for c in df['cluster_label'].unique():
    subset = df[df['cluster_label'] == c]
    plt.scatter(subset['pc1'], subset['pc2'], label='Cluster ' + str(c))

plt.title('Customer Segments (PCA)')
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.legend()
plt.show()